# Chapitre 5 — Fine-tuning de GMIC

On adapte les poids NYU à RSNA. Comme RSNA contient **peu de cancers** (~2 %), un
fine-tuning complet de ~14 M de paramètres **surapprend** vite. La parade =
**linear-probe** : on **gèle les deux backbones** (`ds_net`, `dn_resnet`) et on
n'entraîne que la **tête** (~0,13 M params, < 1 %).

| Régime | On entraîne | Quand | Risque |
|---|---|---|---|
| Full fine-tuning | tout | beaucoup de données cible | surapprentissage |
| **Linear-probe** (ce chapitre) | la tête seule | peu de positifs | sous-apprentissage si domaine trop loin |
| Gradual unfreezing | tête, puis backbone par paliers | compromis | plus de réglages |

Les **démos jouet** ci-dessous tournent sur CPU avec `torch` seul. La
démonstration du gel tourne sur le **vrai modèle GMIC**. La boucle complète (qui
exige les données RSNA + GPU) est donnée en référence à la fin.

In [ ]:
import os, sys
import torch
from course_utils import flowchart
sys.path.insert(0, os.path.expanduser('~/GMIC'))

flowchart([
    '1. Charger le backbone pre-entraine (strict=False)',
    '2. Geler ds_net + dn_resnet (requires_grad=False)',
    '3. Figer la BatchNorm du backbone (.eval())',
    '4. N optimiser que la tete (loss 3 tetes)',
], title='Les 4 gestes du linear-probe')

## Geste 1 — Charger les poids (`strict=False`)

`load_state_dict(strict=False)` tolère les **clés manquantes** (paramètre du modèle
absent du checkpoint → garde son init aléatoire, typiquement la tête redimensionnée)
et les **clés en trop** (ignorées). C'est exactement ce qu'on veut : réutiliser le
backbone, réinitialiser la tête. ⚠️ Il ne tolère **pas** les *shapes* incompatibles.

In [ ]:
import torch.nn as nn

# Démo jouet : checkpoint sans 'head.weight' -> reste à l'init
ckpt = {'feat.weight': torch.randn(4, 1), 'feat.bias': torch.randn(4), 'head.bias': torch.randn(2)}
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.feat = nn.Linear(1, 4)
        self.head = nn.Linear(4, 2)
    def forward(self, x): return self.head(torch.relu(self.feat(x)))
missing, unexpected = Net().load_state_dict(ckpt, strict=False)
print('manquantes :', missing)      # ['head.weight'] -> init aléatoire
print('en trop    :', unexpected)   # []

## Geste 2 — Geler le backbone (sur le vrai GMIC)

Les deux gros extracteurs ResNet sont enregistrés sous les préfixes **`ds_net`**
(global) et **`dn_resnet`** (local). Mettre `requires_grad=False` sur ces préfixes :
(1) aucun gradient calculé, (2) exclus de l'optimiseur, (3) moins de mémoire. On le
vérifie sur le modèle réel.

In [ ]:
from src.modeling.gmic import GMIC

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
params = {'device_type': 'gpu' if DEVICE.type == 'cuda' else 'cpu', 'gpu_number': 0,
          'cam_size': (46, 30), 'K': 6, 'crop_shape': (256, 256), 'percent_t': 0.02,
          'use_v1_global': False, 'post_processing_dim': 256, 'num_classes': 2}
model = GMIC(params)
ckpt = torch.load(os.path.expanduser('~/GMIC/models/sample_model_1.p'),
                  map_location='cpu', weights_only=False)
model.load_state_dict(ckpt, strict=False)

BACKBONE = ('ds_net', 'dn_resnet')
def freeze_backbone(model):
    n = 0
    for name, p in model.named_parameters():
        if name.startswith(BACKBONE):
            p.requires_grad = False
            n += p.numel()
    return n

frozen = freeze_backbone(model)
head = sum(p.numel() for _, p in model.named_parameters() if p.requires_grad)
print(f'backbone gelé : {frozen/1e6:.2f} M params (requires_grad=False)')
print(f'tête entraînable : {head/1e6:.2f} M params  ({100*head/(head+frozen):.1f} %)')

## Geste 3 — Le piège **BatchNorm**

`requires_grad=False` gèle les **poids**, mais **pas** les statistiques de
BatchNorm : en mode `train()`, `running_mean`/`running_var` continuent de se mettre
à jour à chaque batch — le backbone « gelé » dériverait quand même. Il faut mettre
ses modules BN en `.eval()` (et le **ré-appliquer après chaque `model.train()`**,
qui est récursif).

In [ ]:
bn = nn.BatchNorm1d(3)
bn.train()
for _ in range(5):
    bn(torch.randn(16, 3) + 10.0)
print('train() -> running_mean dérive :', [round(v, 2) for v in bn.running_mean.tolist()])
bn.eval()
frozen_mean = bn.running_mean.clone()
for _ in range(5):
    bn(torch.randn(16, 3) + 100.0)
print('eval()  -> figée ?', torch.allclose(frozen_mean, bn.running_mean))

def set_backbone_eval(model):
    for name, module in model.named_modules():
        if name.startswith(BACKBONE):
            module.eval()
set_backbone_eval(model)
print('BN du backbone GMIC figée.')

## Geste 4 — La loss 3 têtes (et le piège `BCEWithLogits`)

La loss du papier supervise les **trois** sorties + régularise la saillance :

$$\mathcal{L} = \mathrm{BCE}(y_{fusion}) + \mathrm{BCE}(y_{global}) + \mathrm{BCE}(y_{local}) + \beta\,\lVert \mathrm{saliency}\rVert_1$$

Les sorties GMIC sont **déjà des probabilités** (sigmoïde/agrégation bornée). On
utilise donc `binary_cross_entropy`, **jamais** `BCEWithLogits` qui ré-appliquerait
une sigmoïde (double sigmoïde → gradients écrasés → collapse).

In [ ]:
import torch.nn.functional as F
proba = torch.tensor([0.99, 0.01])     # DÉJÀ une proba
target = torch.tensor([1.0, 0.0])
good = F.binary_cross_entropy(proba.clamp(1e-7, 1 - 1e-7), target)
bad = F.binary_cross_entropy_with_logits(proba, target)
print(f'BCE(proba)        = {good.item():.4f}  (correct : prédiction sûre et juste)')
print(f'BCEWithLogits     = {bad.item():.4f}  (FAUX : retraite 0.99 en sigmoid(0.99)=0.73)')

## Le vrai code (référence)

Pour superviser les 3 têtes, le projet sous-classe `GMIC` en `GMICTrain` (forward qui
renvoie `y_fusion, y_global, y_local, saliency_map` + rotation aléatoire des patchs
en train). La boucle assemble les 4 gestes. Extrait (non exécuté ici — exige les
données RSNA + GPU) :

```python
EPS = 1e-7
def bce(prob, target):
    return F.binary_cross_entropy(prob.clamp(EPS, 1 - EPS), target)

def compute_loss(yf, yg, yl, sal, target, beta):
    return (bce(yf.float(), target) + bce(yg.float(), target)
            + bce(yl.float(), target) + beta * sal.float().abs().mean())

optimizer = torch.optim.Adam(
    [{'params': [p for n, p in model.named_parameters()
                 if p.requires_grad and not n.startswith(BACKBONE)], 'lr': 1e-4}],
    weight_decay=1e-5)

for epoch in range(epochs):
    model.train()
    set_backbone_eval(model)               # re-fige la BN APRÈS train()
    for x, y in train_loader:
        x = gpu_standardize(x.to(device))
        with torch.autocast('cuda'):
            yf, yg, yl, sal = model(x)
        loss = compute_loss(yf, yg, yl, sal, y.to(device), beta=2e-4)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    auc = evaluate(model, val_loader)      # AUC reportée PAR SEIN (pas par vue)
```

Commande de lancement réelle (sur la VM) :

```bash
python -m fine_tuning.train_gmic \
    --weights GMIC/models/sample_model_1.p \
    --freeze-backbone --percent-t 0.02 \
    --epochs 30 --batch-size 16 --lr-head 1e-4 \
    --out checkpoints/gmic_ft_frozen
```

## Saturer le GPU : augmentation et normalisation sur GPU (`--gpu-augment`)

GMIC s'entraîne en **pleine résolution 2944×1920**. Sur la VM (GPU L40S puissant mais seulement **8 vCPU**), le poste coûteux est le **décodage des PNG** côté CPU. Si les workers du `DataLoader` doivent EN PLUS augmenter les images, ils ne suivent pas → le GPU attend et plafonne à ~**49 %** : *CPU/IO-bound*, pas *GPU-bound*.

La parade (flag `--gpu-augment`, module `fine_tuning/augment.py`) : les workers CPU ne font plus que **décoder** ; l'**augmentation** (affine + gamma) ET la **normalisation z-score** sont appliquées au **batch entier sur le GPU** via kornia, juste après le transfert. Résultat : utilisation GPU **49 % → 91 %**, entraînement nettement plus rapide.

Spécificités GMIC : **pas de flip horizontal** (les vues sont déjà normalisées chest-wall-à-gauche), rotation de la ROI {0, 90, 180, 270} faite dans le `forward`. Le principe général est expliqué au **chapitre 3** ; voici le code réel :

In [ ]:
import torch, torch.nn as nn
import kornia.augmentation as K

GMIC_H, GMIC_W = 2944, 1920

class GPUAugment(nn.Module):
    """Extrait réel de fine_tuning/augment.py. affine (rot ±15°, scale 0.9-1.1,
    translation ±48/32 px) + gamma 0.8-1.25. Normalisation par image avant gamma."""
    def __init__(self, input_size=(GMIC_H, GMIC_W), max_shift=(48, 32)):
        super().__init__()
        tx, ty = max_shift[1] / input_size[1], max_shift[0] / input_size[0]
        self.affine = K.RandomAffine(degrees=15.0, translate=(tx, ty),
                                     scale=(0.9, 1.1), p=1.0, padding_mode='zeros')
        self.gamma = K.RandomGamma(gamma=(0.8, 1.25), p=0.5)
    def forward(self, x):
        mx = x.amax(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)   # normalise par image
        x = x / mx
        x = self.affine(x)
        return self.gamma(x.clamp(0.0, 1.0))

def gpu_standardize(x):
    """z-score par image, sur GPU (la normalisation quitte aussi le CPU)."""
    x = x.float()
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    s = x.std(dim=(1, 2, 3), keepdim=True, unbiased=False).clamp(min=1e-5)
    return (x - m) / s

device = 'cuda' if torch.cuda.is_available() else 'cpu'
aug = GPUAugment().to(device)
batch = torch.rand(2, 1, 736, 480, device=device)   # batch jouet (même code qu'en 2944×1920)
out = gpu_standardize(aug(batch))
print('device:', device, '| batch', tuple(batch.shape), '-> augmenté + standardisé', tuple(out.shape))
print(f'après z-score : moyenne ≈ {out.mean().item():.1e}, écart-type ≈ {out.std().item():.2f}')

## Leçons du projet (ce qui marche, ce qui ne marche pas)

- **Reporter l'AUC par SEIN**, pas par vue : agréger les probas des vues CC+MLO d'un
  même côté ajoute **+0,04 à +0,05** d'AUC « gratuitement ».
- **Le gel stabilise** (pas d'oubli des features NYU) mais **plafonne** (~0,85 par
  sein). Le full fine-tuning monte plus haut (~0,89) mais surapprend dès l'epoch 1
  → garder un **early-stopping** sur l'AUC par sein.
- **Le bottleneck n'est pas la régularisation** (dropout / weight-decay / aug) mais
  l'**échelle des données** (~500 examens malins). Plafond ~**0,886** par sein,
  robuste sur toutes les configs testées.
- **L'ensemble** des 5 modèles NYU casse le plateau et dépasse **0,90**.

Chapitre suivant → `06_risk_control_abstention.ipynb` : transformer un score en une
**garantie** de risque.

## 🧪 Smoke test

Vérifie que le code clé de ce chapitre tourne **sans télécharger les données ni lancer le preprocessing complet** (entrées synthétiques).

In [ ]:
# 🧪 Smoke test — valide gel backbone, BN figée et loss 3 têtes sur un modèle JOUET (aucun poids GMIC).
import torch, torch.nn as nn, torch.nn.functional as F

BACKBONE = ('ds_net', 'dn_resnet')
class _Toy(nn.Module):
    def __init__(self):
        super().__init__()
        self.ds_net = nn.Sequential(nn.Linear(4, 4), nn.BatchNorm1d(4))   # backbone
        self.dn_resnet = nn.Linear(4, 4)                                  # backbone
        self.head = nn.Linear(4, 3)                                       # tête
_toy = _Toy()

# 1. Gel par préfixe : seule la tête reste entraînable
for _n, _p in _toy.named_parameters():
    if _n.startswith(BACKBONE):
        _p.requires_grad = False
assert all(not _p.requires_grad for _n, _p in _toy.named_parameters() if _n.startswith(BACKBONE))
assert all(_p.requires_grad for _n, _p in _toy.named_parameters() if _n.startswith('head'))

# 2. BatchNorm du backbone figée (.eval())
for _n, _mod in _toy.named_modules():
    if _n.startswith(BACKBONE):
        _mod.eval()
assert not _toy.ds_net[1].training, 'la BN du backbone devrait être en eval()'

# 3. Loss 3 têtes : BCE sur des PROBAS sigmoïdées (PAS BCEWithLogits) + terme L1
_EPS = 1e-7
_probs = [torch.sigmoid(torch.randn(8, 1, requires_grad=True)) for _ in range(3)]
_tgts = [torch.randint(0, 2, (8, 1)).float() for _ in range(3)]
_sal = torch.rand(8, 1, requires_grad=True)
_loss = sum(F.binary_cross_entropy(_p.clamp(_EPS, 1 - _EPS), _t) for _p, _t in zip(_probs, _tgts)) \
        + 0.001 * _sal.abs().mean()
_loss.backward()
assert torch.isfinite(_loss), _loss
print(f'✅ Gel backbone + BN figée + loss 3 têtes (BCE sur sigmoïde + L1) OK | loss={_loss.item():.3f}')